# Generación de árboles no isomorfos

Se enumeran los árboles de orden $n$ y se calculan sus invariantes $S(T)$, $Z_1(T)$ y $W(T)$. Los registros se almacenan en SQLite para su análisis posterior.

## 1. Configuración

Se fija un único orden y se establece el directorio de almacenamiento.

In [ ]:
import sqlite3
from pathlib import Path

import networkx as nx
import numpy as np

N = 23
TOL_CERO = 1e-12
TAMANO_LOTE = 10000
PROGRESO_CADA = 100000

CARPETA = Path(f"resultados_seminario_orden_{N}") / "datos"
CARPETA.mkdir(parents=True, exist_ok=True)
RUTA_SQLITE = CARPETA / f"arboles_orden_{N}.sqlite"

print(f"Orden: {N}")
print(f"Base de datos: {RUTA_SQLITE.resolve()}")

## 2. Invariantes

Para un árbol $T$ de orden $n$ se emplean

$$
S(T)=-\sum_{p_i>0}p_i\log_2p_i,\qquad p_i=\frac{\lambda_i}{2(n-1)},
$$

$$
Z_1(T)=\sum_{v\in V(T)}d(v)^2,\qquad W(T)=\sum_{\{u,v\}\subseteq V(T)}d(u,v).
$$

El índice de Wiener se obtiene mediante las contribuciones $s(n-s)$ de cada arista, donde $s$ es el tamaño de una de las componentes al retirarla.

In [ ]:
def calcular_invariantes(T):
    n = T.number_of_nodes()
    L = np.zeros((n, n), dtype=float)

    for v, grado in T.degree():
        L[v, v] = grado
    for u, v in T.edges():
        L[u, v] = -1.0
        L[v, u] = -1.0

    autovalores = np.linalg.eigvalsh(L)
    autovalores[np.abs(autovalores) < TOL_CERO] = 0.0
    p = autovalores / (2 * (n - 1))
    p = p[p > TOL_CERO]
    entropia = float(-np.sum(p * np.log2(p)))

    zagreb = sum(grado**2 for _, grado in T.degree())

    padre = {0: None}
    orden = [0]
    for u in orden:
        for v in T.neighbors(u):
            if v != padre[u]:
                padre[v] = u
                orden.append(v)

    tamanos = {v: 1 for v in orden}
    wiener = 0
    for v in reversed(orden[1:]):
        s = tamanos[v]
        wiener += s * (n - s)
        tamanos[padre[v]] += s

    return entropia, int(zagreb), int(wiener)


def codigo_graph6(T):
    return nx.to_graph6_bytes(T, header=False).decode("ascii").strip()

## 3. Generación y almacenamiento

Los datos se escriben por lotes. Una base parcial puede reanudarse a partir del último índice consecutivo almacenado.

In [ ]:
def generar_base(n, ruta):
    esperado = nx.number_of_nonisomorphic_trees(n)
    conexion = sqlite3.connect(ruta)

    try:
        conexion.execute("""
            CREATE TABLE IF NOT EXISTS arboles (
                indice INTEGER PRIMARY KEY,
                codigo TEXT NOT NULL,
                entropia REAL NOT NULL,
                zagreb1 INTEGER NOT NULL,
                wiener INTEGER NOT NULL
            )
        """)
        conexion.execute("""
            CREATE TABLE IF NOT EXISTS metadata (
                clave TEXT PRIMARY KEY, valor TEXT NOT NULL
            )
        """)
        fila = conexion.execute(
            "SELECT valor FROM metadata WHERE clave='n'"
        ).fetchone()
        if fila is not None and int(fila[0]) != n:
            raise ValueError("El archivo corresponde a otro orden.")
        conexion.execute(
            "INSERT OR REPLACE INTO metadata VALUES ('n', ?)", (str(n),)
        )
        conexion.commit()

        cantidad, minimo, maximo = conexion.execute(
            "SELECT COUNT(*), MIN(indice), MAX(indice) FROM arboles"
        ).fetchone()
        inicio = 0 if cantidad == 0 else maximo + 1
        if cantidad > esperado or (cantidad and (minimo != 0 or cantidad != inicio)):
            raise ValueError("Los índices de la base no son consecutivos.")

        if cantidad < esperado:
            print(f"Registros existentes: {cantidad:,} de {esperado:,}")
            lote = []
            for indice, T in enumerate(nx.nonisomorphic_trees(n)):
                if indice < inicio:
                    continue
                S, Z, W = calcular_invariantes(T)
                lote.append((indice, codigo_graph6(T), S, Z, W))

                if len(lote) >= TAMANO_LOTE:
                    conexion.executemany(
                        "INSERT INTO arboles VALUES (?, ?, ?, ?, ?)", lote
                    )
                    conexion.commit()
                    lote.clear()

                if PROGRESO_CADA and (indice + 1) % PROGRESO_CADA == 0:
                    print(f"Árboles procesados: {indice+1:,} / {esperado:,}")

            if lote:
                conexion.executemany(
                    "INSERT INTO arboles VALUES (?, ?, ?, ?, ?)", lote
                )
                conexion.commit()

        cantidad = conexion.execute("SELECT COUNT(*) FROM arboles").fetchone()[0]
        if cantidad != esperado:
            raise RuntimeError(f"Base incompleta: {cantidad:,} de {esperado:,} registros.")

        conexion.execute(
            "CREATE INDEX IF NOT EXISTS idx_entropia ON arboles(entropia, indice)"
        )
        conexion.execute(
            "INSERT OR REPLACE INTO metadata VALUES ('completa', '1')"
        )
        conexion.commit()
        print(f"Generación completa: {cantidad:,} árboles de orden {n}.")

    finally:
        conexion.close()

## 4. Ejecución

In [ ]:
generar_base(N, RUTA_SQLITE)